# MetaPulsar Usage Example

This notebook demonstrates how to use MetaPulsar to combine pulsar timing data from multiple PTA collaborations into unified Enterprise pulsar objects.

## Overview

The workflow covers:
1. **Manual single-pulsar data preparation** - Creating data structures by hand
2. **MetaPulsar creation** - Using consistent parameter merging strategies
3. **Enterprise integration** - Working with the resulting pulsar objects
4. **Automated discovery** - Processing multiple pulsars automatically
5. **Reference PTA selection** - Different strategies for choosing reference PTAs

## Key Concepts

- **Reference PTA**: The PTA whose parameters are inherited by the MetaPulsar for merged model components
- **Consistent Strategy**: Merges compatible parameters from different PTAs where possible
- **Component Merging**: Controls which parameter types (astrometry, spindown, binary, dispersion) are merged
- **Parameter Naming**: Merged parameters have no suffix, PTA-specific parameters retain PTA suffixes


In [1]:
import logging
from metapulsar import (
    MetaPulsarFactory,
    discover_files,
    discover_layout,
    combine_layouts,
)

# Suppress debug output for cleaner example
logging.getLogger("metapulsar").setLevel(logging.WARNING)


/opt/venvs/pta/lib/python3.10/site-packages/enterprise/signals/utils.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import Requirement, resource_filename


## Part 1: Manual Single-Pulsar Data Preparation

For single pulsars, manually creating the data structure is often the most transparent and flexible approach. This gives you full control over which PTAs to include and their ordering.

### Data Structure

The data structure is a dictionary where:
- **Keys**: PTA names (e.g., 'nanograv_9y', 'epta_dr2')
- **Values**: Lists of file information dictionaries
- **Reference PTA**: The first PTA in the dictionary becomes the reference PTA

### File Information Fields
- `par`: Path to .par file (pulsar parameters)
- `tim`: Path to .tim file (timing observations)  
- `timespan_days`: Data span in days
- `timing_package`: Software used (tempo2/pint)


In [ ]:
# Manually create a single-pulsar dictionary with three PTAs
# The reference PTA (first in the dictionary) will be used for parameter
# inheritance where appropriate
pulsar_data = {
    # Reference PTA - parameters from this PTA will be inherited by the MetaPulsar
    # for model components that are merged (astrometry, spindown, binary, dispersion)
    "epta_dr2": [
        {
            "par": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par",
            "tim": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim",
            "timing_package": "tempo2",  # Timing package used
        }
    ],
    "nanograv_9y": [
        {
            "par": "../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par",
            "tim": "../../data/ipta-dr2/NANOGrav_9y/tim/J0613-0200_NANOGrav_9yv1.tim",
            "timing_package": "pint",
        }
    ],
    # Additional PTAs - parameters from these will be merged where requested
    "ppta_dr2": [
        {
            "par": "../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par",
            "tim": "../../data/ipta-dr2/PPTA_dr1dr2/tim/J0613-0200_dr1dr2.tim",
            "timing_package": "tempo2",
        }
    ],
}

print("Single-pulsar data structure:")
print("  Pulsar: J0613-0200")
print(f"  PTAs: {list(pulsar_data.keys())}")
print(f"  Reference PTA: {list(pulsar_data.keys())[0]} (first in dictionary)")
print("  Data structure fields:")
print("    - par: Path to .par file (pulsar parameters)")
print("    - tim: Path to .tim file (timing observations)")
print("    - timespan_days: Data span in days")
print("    - timing_package: Software used (tempo2/pint)")


Single-pulsar data structure:
  Pulsar: J0613-0200
  PTAs: ['epta_dr2', 'nanograv_9y', 'ppta_dr2']
  Reference PTA: epta_dr2 (first in dictionary)
  Data structure fields:
    - par: Path to .par file (pulsar parameters)
    - tim: Path to .tim file (timing observations)
    - timespan_days: Data span in days
    - timing_package: Software used (tempo2/pint)


## Part 2: MetaPulsar Creation with Consistent Strategy

The `consistent` strategy merges parameters from different PTAs where possible, inheriting values from the reference PTA for merged model components.

### Combination Strategy Options
- `consistent`: Merge compatible parameters, inherit reference PTA values
- `independent`: Keep all parameters separate with PTA suffixes

### Mergeable Components
- `astrometry`: Position, proper motion, parallax
- `spindown`: Spin frequency and derivatives
- `binary`: Binary orbital parameters
- `dispersion`: Dispersion measure and derivatives


In [3]:
# Create MetaPulsarFactory instance
factory = MetaPulsarFactory()

# Create MetaPulsar using the 'consistent' strategy
# This merges parameters from different PTAs where possible
metapulsar = factory.create_metapulsar(
    file_data=pulsar_data,
    combination_strategy="consistent",  # Merge compatible parameters
    combine_components=[
        "astrometry",
        "spindown",
        "binary",
        "dispersion",
    ],  # Components to merge
    add_dm_derivatives=True,  # Ensure DM1, DM2 are present
)

print(f"Created MetaPulsar: {metapulsar.name}")
print(f"Reference PTA: {list(pulsar_data.keys())[0]}")
print("Combination strategy: consistent")
print("Components merged: astrometry, spindown, binary, dispersion")


2025-10-08 07:47:42.368 | INFO     | metapulsar.metapulsar_factory:create_metapulsar:136 - Creating MetaPulsar using consistent strategy
2025-10-08 07:47:42.373 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for epta_dr2 from ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par
2025-10-08 07:47:42.379 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for nanograv_9y from ../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par
2025-10-08 07:47:42.381 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for ppta_dr2 from ../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par
2025-10-08 07:47:42.381 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 1 files for PTA epta_dr2
2025-10-08 07:47:42.409 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found puls

Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/JBO.DFB.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/JBO.DFB.1520.tim
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.2639.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613

2025-10-08 07:49:14.096 | DEBUG    | pint.toa:__init__:1377 - No pulse number flags found in the TOAs
2025-10-08 07:49:14.115 | DEBUG    | pint.toa:apply_clock_corrections:2224 - Applying clock corrections (include_bipm = True)
2025-10-08 07:49:14.565 | INFO     | pint.observatory:gps_correction:235 - Applying GPS to UTC clock correction (~few nanoseconds)
2025-10-08 07:49:14.566 | DEBUG    | pint.observatory:_load_gps_clock:109 - Loading global GPS clock file
2025-10-08 07:49:14.573 | DEBUG    | pint.observatory.clock_file:__init__:812 - Global clock file gps2utc.clk saving kwargs={'bogus_last_correction': False, 'valid_beyond_ends': False}
2025-10-08 07:49:14.577 | DEBUG    | pint.observatory.clock_file:read_tempo2_clock_file:463 - Loading TEMPO2-format observatory clock correction file gps2utc.clk (/home/vscode/.astropy/cache/download/url/d3c81b5766f4bfb84e65504c8a453085/contents) with bogus_last_correction=False
2025-10-08 07:49:14.597 | INFO     | pint.observatory:find_clock_file:

[preProcess.C:158] Warning: PSR J0613-0200 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



2025-10-08 07:49:30.665 | DEBUG    | pint.models.absolute_phase:get_TZR_toa:101 - Creating and dealing with the single TZR_toa for absolute phase
2025-10-08 07:49:30.667 | DEBUG    | pint.toa:__init__:1377 - No pulse number flags found in the TOAs
2025-10-08 07:49:30.668 | DEBUG    | pint.toa:apply_clock_corrections:2224 - Applying clock corrections (include_bipm = True)
2025-10-08 07:49:30.696 | INFO     | pint.observatory:gps_correction:235 - Applying GPS to UTC clock correction (~few nanoseconds)
2025-10-08 07:49:30.698 | INFO     | pint.observatory:bipm_correction:250 - Applying TT(TAI) to TT(BIPM2023) clock correction (~27 us)
2025-10-08 07:49:30.701 | INFO     | pint.observatory.topo_obs:clock_corrections:340 - Applying observatory clock corrections for observatory='gbt'.
2025-10-08 07:49:30.702 | DEBUG    | pint.toa:compute_TDBs:2270 - Computing TDB columns.
2025-10-08 07:49:30.704 | DEBUG    | pint.toa:compute_TDBs:2291 - Using EPHEM = DE421 for TDB calculation.
2025-10-08 07:4



Results for PSR J0613-0200


RMS pre-fit residual = 0.000 (us), RMS post-fit residual = 63.263 (us)
Fit Chisq = 0	Chisqr/nfree = 0.00/0 = nan	pre/post = 0
Number of fit parameters: 0
Number of points in fit = 0
Offset: 0 1 offset_e*sqrt(n) = 0 n = 0


PARAMETER       Pre-fit                   Post-fit                  Uncertainty   Difference   Fit
---------------------------------------------------------------------------------------------------
RAJ (rad)       1.63071752891066          1.63071752891066          1.7211e-10    0             Y
RAJ (hms)       06:13:43.9756763           06:13:43.9756763         2.3666e-06    0            
DECJ (rad)      -0.0351355399976385       -0.0351355399976385       4.2047e-10    0             Y
DECJ (dms)      -02:00:47.22535           -02:00:47.22535           8.6728e-05    0            
F0 (s^-1)       326.60056202349           326.60056202349           4.3707e-13    0             Y
F1 (s^-2)       -1.02294583448114e-15     -1.02294583448114e-

2025-10-08 07:49:59.530 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-08 07:49:59.702 | WARNING  | pint.models.model_builder:choose_binary_model:627 - Found T2 binary model. Gracefully converting T2 to: ELL1.
2025-10-08 07:49:59.702 | DEBUG    | pint.models.model_builder:convert_binary_params_dict:1056 - Requested to convert binary model for BINARY model: {binary}
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/timing_model.py:400: UserWarning: PINT does not support 'DILATEFREQ Y'
  warn("PINT does not support 'DILATEFREQ Y'")
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/timing_model.py:403: UserWarning: PINT only supports 'TIMEEPH FB90'
  warn("PINT only supports 'TIMEEPH FB90'")
2025-10-08 07:49:59.711 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `Timing

Created MetaPulsar: J0613-0200
Reference PTA: epta_dr2
Combination strategy: consistent
Components merged: astrometry, spindown, binary, dispersion


## Part 3: MetaPulsar as Enterprise Pulsar

The resulting MetaPulsar is a fully functional Enterprise pulsar with all standard attributes. It combines data from multiple PTAs into a single unified object.

### Key Features
- **Unified TOAs**: All timing observations combined
- **Merged Parameters**: Compatible parameters inherit reference PTA values
- **PTA-Specific Parameters**: Incompatible parameters retain PTA suffixes
- **Enterprise Compatibility**: Works with all Enterprise analysis tools


In [5]:
# The resulting MetaPulsar is an Enterprise pulsar with all standard attributes
print("MetaPulsar Enterprise attributes:")
print(f"  Name: {metapulsar.name}")
print(f"  Number of pulsars: {len(metapulsar._pulsars)}")
print(f"  PTA names: {list(metapulsar._pulsars.keys())}")
print(f"  Combination strategy: {metapulsar.combination_strategy}")
print(f"  Components merged: {metapulsar.combine_components}")

# Show some basic Enterprise pulsar attributes
print("\nEnterprise pulsar attributes:")
print(f"  Number of TOAs: {len(metapulsar.toas)}")
print(
    f"  Frequency range: {metapulsar.freqs.min():.2f} - {metapulsar.freqs.max():.2f} MHz"
)
print(f"  Time span: {metapulsar.toas.max() - metapulsar.toas.min():.1f} days")

print(
    "\nThe MetaPulsar combines data from multiple PTAs into a single Enterprise pulsar."
)
print(
    f"Merged parameters inherit values from the reference PTA ({list(pulsar_data.keys())[0]})."
)
print("PTA-specific parameters retain their original PTA-specific values.")


MetaPulsar Enterprise attributes:
  Name: J0613-0200
  Number of pulsars: 3
  PTA names: ['epta_dr2', 'nanograv_9y', 'ppta_dr2']
  Combination strategy: consistent
  Components merged: ['astrometry', 'spindown', 'binary', 'dispersion']

Enterprise pulsar attributes:
  Number of TOAs: 9201
  Frequency range: 323.72 - 3102.14 MHz
  Time span: 506639509.7 days

The MetaPulsar combines data from multiple PTAs into a single Enterprise pulsar.
Merged parameters inherit values from the reference PTA (epta_dr2).
PTA-specific parameters retain their original PTA-specific values.


In [6]:
metapulsar.fitpars

['PX',
 'RAJ',
 'DECJ',
 'PMRA',
 'PMDEC',
 'F0',
 'F1',
 'DM',
 'DM1',
 'DM2',
 'PB',
 'PBDOT',
 'A1',
 'TASC',
 'EPS1',
 'EPS2',
 'FD1_nanograv_9y',
 'FD2_nanograv_9y',
 'JUMP1_nanograv_9y',
 'JUMP1_epta_dr2',
 'JUMP2_epta_dr2',
 'JUMP3_epta_dr2',
 'JUMP4_epta_dr2',
 'JUMP5_epta_dr2',
 'JUMP6_epta_dr2',
 'JUMP7_epta_dr2',
 'JUMP8_epta_dr2',
 'JUMP9_epta_dr2',
 'JUMP10_epta_dr2',
 'JUMP11_epta_dr2',
 'JUMP12_epta_dr2',
 'JUMP13_epta_dr2',
 'JUMP1_ppta_dr2',
 'JUMP2_ppta_dr2',
 'JUMP3_ppta_dr2',
 'JUMP4_ppta_dr2',
 'JUMP5_ppta_dr2',
 'JUMP6_ppta_dr2',
 'JUMP7_ppta_dr2',
 'JUMP8_ppta_dr2',
 'JUMP9_ppta_dr2',
 'JUMP10_ppta_dr2',
 'JUMP11_ppta_dr2',
 'JUMP12_ppta_dr2',
 'JUMP13_ppta_dr2']

In [8]:
metapulsar._designmatrix.shape

(9201, 45)

### Parameter Naming Conventions

MetaPulsar uses a clear naming convention to distinguish between merged and PTA-specific parameters:

- **Merged parameters**: No suffix, inherit reference PTA values
- **PTA-specific parameters**: Retain PTA suffix (e.g., `_nanograv_9y`, `_epta_dr2`)

This allows you to easily identify which parameters are shared across PTAs and which are specific to individual PTAs.


In [9]:
# Demonstrate parameter naming conventions
print("Parameter naming conventions:")
print("Merged parameters (no suffix):")
fitparams = metapulsar.fitpars
# Get PTA names from our data structure
pta_names = list(pulsar_data.keys())
pta_suffixes = [f"_{pta}" for pta in pta_names]

merged_params = [
    p for p in fitparams if not any(suffix in p for suffix in pta_suffixes)
]
for param in merged_params[:5]:  # Show first 5 merged parameters
    print(f"  {param}")

print("\nPTA-specific parameters (retain PTA suffix):")
pta_specific_params = [
    p for p in fitparams if any(suffix in p for suffix in pta_suffixes)
]
for param in pta_specific_params[:5]:  # Show first 5 PTA-specific parameters
    print(f"  {param}")

print("\nThis naming convention allows you to distinguish between:")
print("  - Merged parameters: Inherit reference PTA values, no suffix")
print("  - PTA-specific parameters: Retain original values, keep PTA suffix")


Parameter naming conventions:
Merged parameters (no suffix):
  PX
  RAJ
  DECJ
  PMRA
  PMDEC

PTA-specific parameters (retain PTA suffix):
  FD1_nanograv_9y
  FD2_nanograv_9y
  JUMP1_nanograv_9y
  JUMP1_epta_dr2
  JUMP2_epta_dr2

This naming convention allows you to distinguish between:
  - Merged parameters: Inherit reference PTA values, no suffix
  - PTA-specific parameters: Retain original values, keep PTA suffix


### Controlling Component Merging

You can control which components are merged by adjusting the `combine_components` parameter. When a component is not merged, its parameters retain PTA-specific suffixes.


In [10]:
# Demonstrate the effect of NOT merging astrometry parameters
print("Example: NOT merging astrometry parameters")
print("-" * 60)

# Create another MetaPulsar without merging astrometry
metapulsar_no_astrometry = factory.create_metapulsar(
    file_data=pulsar_data,
    combination_strategy="consistent",
    combine_components=["spindown", "binary", "dispersion"],  # Exclude astrometry
    add_dm_derivatives=True,
)

print("When astrometry is NOT merged, astrometry parameters get PTA suffixes:")
fitparams_no_astrometry = metapulsar_no_astrometry.fitpars
astrometry_params = [
    p
    for p in fitparams_no_astrometry
    if any(
        term in p.lower()
        for term in [
            "ra", "dec", "pmra", "pmdec", "px", "elong", "elat",
            "pmelong", "pmelat", "beta", "lambda", "pmbeta", "pmlambda",
        ]
    )
]
for param in astrometry_params[:3]:  # Show first 3 astrometry parameters
    print(f"  {param}")

print("\nCompare with merged astrometry (from first example):")
astrometry_merged = [
    p
    for p in fitparams
    if any(
        term in p.lower()
        for term in [
            "ra", "dec", "pmra", "pmdec", "px", "elong", "elat",
            "pmelong", "pmelat", "beta", "lambda", "pmbeta", "pmlambda",
        ]
    )
]
for param in astrometry_merged[:3]:  # Show first 3 astrometry parameters
    print(f"  {param} (no suffix - merged)")

print(f"\nNote: PTA suffixes used are: {', '.join(pta_suffixes)}")

print(
    "\nThis shows how combine_components controls which parameters are merged vs. kept PTA-specific."
)


2025-10-08 07:56:24.662 | INFO     | metapulsar.metapulsar_factory:create_metapulsar:136 - Creating MetaPulsar using consistent strategy
2025-10-08 07:56:24.727 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for epta_dr2 from ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par
2025-10-08 07:56:24.745 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for nanograv_9y from ../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par
2025-10-08 07:56:24.764 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for ppta_dr2 from ../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par


2025-10-08 07:56:24.775 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 1 files for PTA epta_dr2
2025-10-08 07:56:24.810 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0613-0200 at RA=6.2289h, DEC=-2.0131°
2025-10-08 07:56:24.811 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 1 files for PTA nanograv_9y


Example: NOT merging astrometry parameters
------------------------------------------------------------


2025-10-08 07:56:24.925 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0613-0200 at RA=6.2291h, DEC=-2.0132°
2025-10-08 07:56:24.926 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 1 files for PTA ppta_dr2
2025-10-08 07:56:24.927 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0613-0200 at RA=6.2289h, DEC=-2.0131°
2025-10-08 07:56:24.944 | INFO     | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:569 - Discovered 1 unique pulsars across all PTAs
2025-10-08 07:56:24.945 | INFO     | metapulsar.metapulsar_factory:_validate_single_pulsar_data:215 - Validated single pulsar data for: J0613-0200
2025-10-08 07:56:41.021 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-08 07:56:41.273 | WARNING  | pint.models.model_builder:choose_binary

Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/JBO.DFB.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/JBO.DFB.1520.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.2639.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200

...

[preProcess.C:158] Warning: PSR J0613-0200 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



2025-10-08 07:57:33.062 | INFO     | pint.solar_system_ephemerides:_load_kernel_link:85 - Set solar system ephemeris to de421 from download
2025-10-08 07:57:34.993 | DEBUG    | pint.toa:add_vel_ecl:2488 - Adding column ssb_obs_vel_ecl
2025-10-08 07:57:34.994 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-08 07:57:35.029 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-08 07:57:35.360 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-08 07:57:35.377 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-08 07:57:37.493 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-08 07:57:37.577 | DEBUG    | pint.models.absolute_phase:get_TZR_toa:101 - Creating and dealing with the single TZR_toa for absolute phase
2025-10-08 07:57:37



Results for PSR J0613-0200


RMS pre-fit residual = 0.000 (us), RMS post-fit residual = 63.263 (us)
Fit Chisq = 0	Chisqr/nfree = 0.00/0 = nan	pre/post = 0
Number of fit parameters: 0
Number of points in fit = 0
Offset: 0 1 offset_e*sqrt(n) = 0 n = 0


PARAMETER       Pre-fit                   Post-fit                  Uncertainty   Difference   Fit
---------------------------------------------------------------------------------------------------
RAJ (rad)       1.63071752891066          1.63071752891066          1.7211e-10    0             Y
RAJ (hms)       06:13:43.9756763           06:13:43.9756763         2.3666e-06    0            
DECJ (rad)      -0.0351355399976385       -0.0351355399976385       4.2047e-10    0             Y
DECJ (dms)      -02:00:47.22535           -02:00:47.22535           8.6728e-05    0            
F0 (s^-1)       326.60056202349           326.60056202349           4.3707e-13    0             Y
F1 (s^-2)       -1.02294583448114e-15     -1.02294583448114e-

2025-10-08 07:59:05.605 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-08 07:59:05.943 | WARNING  | pint.models.model_builder:choose_binary_model:627 - Found T2 binary model. Gracefully converting T2 to: ELL1.
2025-10-08 07:59:05.944 | DEBUG    | pint.models.model_builder:convert_binary_params_dict:1056 - Requested to convert binary model for BINARY model: {binary}
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/timing_model.py:400: UserWarning: PINT does not support 'DILATEFREQ Y'
  warn("PINT does not support 'DILATEFREQ Y'")
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/timing_model.py:403: UserWarning: PINT only supports 'TIMEEPH FB90'
  warn("PINT only supports 'TIMEEPH FB90'")
2025-10-08 07:59:05.992 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `Timing

When astrometry is NOT merged, astrometry parameters get PTA suffixes:
  PX_nanograv_9y
  ELONG_nanograv_9y
  ELAT_nanograv_9y

Compare with merged astrometry (from first example):
  PX (no suffix - merged)
  RAJ (no suffix - merged)
  DECJ (no suffix - merged)

Note: PTA suffixes used are: _epta_dr2, _nanograv_9y, _ppta_dr2

This shows how combine_components controls which parameters are merged vs. kept PTA-specific.


## Part 4: Automated Multi-Pulsar Processing

For processing multiple pulsars, manually creating data structures becomes cumbersome. MetaPulsar provides utility functions based on regex pattern matching for automation.

### Automated Workflow
1. **Discover layouts**: Automatically detect data release directory structures
2. **Combine layouts**: Merge discovered layouts with predefined patterns
3. **Discover files**: Find all pulsar files using pattern matching
4. **Create MetaPulsars**: Process discovered pulsars automatically (limited to 3 for performance)

### Reference PTA Selection Strategies
- **Auto-selection**: Choose PTA with longest timespan per pulsar
- **Global reference**: Use same PTA as reference for all pulsars
- **Manual**: Reorder PTAs for specific pulsars

**Note**: For demonstration purposes, we limit processing to the first 3 pulsars to keep execution time reasonable.


In [11]:
print("Manually creating data structures for all pulsars in an array is cumbersome.")
print("We provide utility functions based on regex pattern matching for automation.")

# Discover data release layouts
print("\n1. Discovering data release layouts...")
layout1 = discover_layout("../../data/ipta-dr2/EPTA_v2.2")
layout2 = discover_layout("../../data/ipta-dr2/PPTA_dr1dr2")

print("Discovered layout in ../../data/ipta-dr2/EPTA_v2.2:")
for key, value in layout1.items():
    print(f"  - {key} = {value}")

print("\nDiscovered layout in ../../data/ipta-dr2/PPTA_dr1dr2:")
for key, value in layout2.items():
    print(f"  - {key} = {value}")


2025-10-07 20:05:39.462 | INFO     | metapulsar.layout_discovery_service:_analyze_directory_structure:77 - Analyzing directory structure: ../../data/ipta-dr2/EPTA_v2.2
2025-10-07 20:05:39.552 | INFO     | metapulsar.layout_discovery_service:_analyze_directory_structure:90 - Found 42 par files and 338 tim files
2025-10-07 20:05:39.662 | INFO     | metapulsar.layout_discovery_service:_detect_timing_package:325 - Found BINARY T2 in J0034-0534.par - detected tempo2


Manually creating data structures for all pulsars in an array is cumbersome.
We provide utility functions based on regex pattern matching for automation.

1. Discovering data release layouts...
Discovered layout in ../../data/ipta-dr2/EPTA_v2.2:
  - base_dir = '../../data/ipta-dr2/EPTA_v2.2'
  - par_pattern = '([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)\\.par'
  - tim_pattern = '([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)\\.*\\_all.tim'
  - timing_package = 'tempo2'
  - priority = 1
  - description = 'Auto-discovered PTA from ../../data/ipta-dr2/EPTA_v2.2'


2025-10-07 20:05:39.667 | INFO     | metapulsar.layout_discovery_service:_analyze_directory_structure:77 - Analyzing directory structure: ../../data/ipta-dr2/PPTA_dr1dr2
2025-10-07 20:05:39.670 | INFO     | metapulsar.layout_discovery_service:_analyze_directory_structure:90 - Found 18 par files and 57 tim files
2025-10-07 20:05:39.675 | INFO     | metapulsar.layout_discovery_service:_detect_timing_package:325 - Found BINARY T2 in J0437-4715_dr1dr2.par - detected tempo2


Discovered layout in ../../data/ipta-dr2/PPTA_dr1dr2:
  - base_dir = '../../data/ipta-dr2/PPTA_dr1dr2'
  - par_pattern = 'par/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?).*\\.par'
  - tim_pattern = 'par/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?).*\\.*\\_dr1dr2.tim'
  - timing_package = 'tempo2'
  - priority = 1
  - description = 'Auto-discovered PTA from ../../data/ipta-dr2/PPTA_dr1dr2'
Discovered layout in ../../data/ipta-dr2/EPTA_v2.2:
  - discovered_EPTA_v2.2 = {'base_dir': '../../data/ipta-dr2/EPTA_v2.2', 'par_pattern': '([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)\\.par', 'tim_pattern': '([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?)\\.*\\_all.tim', 'timing_package': 'tempo2', 'priority': 1, 'description': 'Auto-discovered PTA from ../../data/ipta-dr2/EPTA_v2.2', 'discovery_confidence': 0.6}

Discovered layout in ../../data/ipta-dr2/PPTA_dr1dr2:
  - discovered_PPTA_dr1dr2 = {'base_dir': '../../data/ipta-dr2/PPTA_dr1dr2', 'par_pattern': 'par/([BJ]\\d{4}[+-]\\d{2,4}[A-Z]?).*\\

In [13]:
# Combine layouts (only use available ones)
# With include_defaults=True we also include a dictionary of default regexp patterns
# for data releases that we have seen before
combined_layout = combine_layouts(layout1, layout2, include_defaults=False)
print(f"   Discovered {len(combined_layout)} data releases")

# Discover files using the combined layout
print("\n2. Discovering files...")
file_data = discover_files(
    working_dir=".",  # Current directory
    pta_data_releases=combined_layout,
    verbose=True,
)

print(f"   Found files for {len(file_data)} data releases")


2025-10-07 20:10:40.502 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.1360.tim
2025-10-07 20:10:40.504 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.1410.tim
2025-10-07 20:10:40.507 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.2639.tim
2025-10-07 20:10:40.509 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/NRT.BON.1400.tim
2025-10-07 20:10:40.515 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/NRT.BON.1600.tim
2025-10-07 20:10:40.518 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processin

   Discovered 2 data releases

2. Discovering files...


2025-10-07 20:10:40.628 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0751+1807/tims/JBO.DFB.1400.tim
2025-10-07 20:10:40.629 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0751+1807/tims/JBO.DFB.1520.tim
2025-10-07 20:10:40.631 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0751+1807/tims/EFF.EBPP.1360.tim
2025-10-07 20:10:40.633 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0751+1807/tims/EFF.EBPP.1410.tim
2025-10-07 20:10:40.636 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processing included TOA file ../../data/ipta-dr2/EPTA_v2.2/J0751+1807/tims/EFF.EBPP.2639.tim
2025-10-07 20:10:40.638 | DEBUG    | metapulsar.tim_file_analyzer:_handle_command:143 - Processin

Found:
  - discovered_EPTA_v2.2: 42 pulsars
  (No pulsars for: discovered_PPTA_dr1dr2)
   Found files for 2 data releases


# Looks like the PPTA discovery is not working as expected yet!

In [17]:
from metapulsar import get_pulsar_names_from_file_data, filter_file_data_by_pulsars

# Create MetaPulsars for all discovered pulsars (limited to 3 for performance)
print("\n3. Creating MetaPulsars for a subset of pulsars...")

# Use coordinate-based discovery to get actual pulsar names
print("   Using coordinate-based discovery to identify actual pulsars...")
pulsar_names = get_pulsar_names_from_file_data(file_data)
limited_pulsar_names = pulsar_names[:3]
print(
    f"   Found {len(pulsar_names)} pulsars, limiting to first {len(limited_pulsar_names)} for performance: {limited_pulsar_names}"
)

# Filter file_data to only include the limited pulsars using coordinate-based grouping
limited_file_data = filter_file_data_by_pulsars(file_data, limited_pulsar_names)

2025-10-07 20:14:27.926 | INFO     | metapulsar.metapulsar_factory:group_files_by_pulsar:244 - Grouping files by pulsar using coordinate-based identification
2025-10-07 20:14:27.932 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 42 files for PTA discovered_EPTA_v2.2
2025-10-07 20:14:27.954 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0030+0451 at RA=0.5076h, DEC=4.8610°
2025-10-07 20:14:27.958 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0034-0534 at RA=0.5727h, DEC=-5.5769°
2025-10-07 20:14:27.977 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0218+4232 at RA=2.3018h, DEC=42.5382°
2025-10-07 20:14:27.979 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0610-2100 at RA=6.1704h, DEC=-21.0078°
2025-10-07 20:14:27.


3. Creating MetaPulsars for a subset of pulsars...
   Using coordinate-based discovery to identify actual pulsars...


2025-10-07 20:14:28.156 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J1802-2124 at RA=18.0348h, DEC=-21.4010°
2025-10-07 20:14:28.162 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J1804-2717 at RA=18.0725h, DEC=-27.2920°
2025-10-07 20:14:28.175 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J1843-1113 at RA=18.7281h, DEC=-11.2253°
2025-10-07 20:14:28.188 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J1853+1303 at RA=18.8993h, DEC=13.0622°
2025-10-07 20:14:28.210 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J1857+0943 at RA=18.9601h, DEC=9.7214°
2025-10-07 20:14:28.224 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J1909-3744 at RA=19.1632h, DEC=-37.7374°
2

   Found 42 pulsars, limiting to first 3 for performance: ['J0030+0451', 'J0034-0534', 'J0218+4232']


In [18]:
# Option 1: Auto-select reference PTA by timespan for each pulsar
print("   Option 1: Auto-select reference PTA by timespan (per pulsar)")
metapulsars_auto = factory.create_all_metapulsars(
    file_data=limited_file_data,
    combination_strategy="consistent",
    reference_pta=None,  # Auto-select by longest timespan
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
)

print(
    f"   Created {len(metapulsars_auto)} MetaPulsars with auto-selected reference PTAs"
)

2025-10-07 20:14:39.666 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 3 files for PTA discovered_EPTA_v2.2
2025-10-07 20:14:39.671 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0030+0451 at RA=0.5076h, DEC=4.8610°
2025-10-07 20:14:39.673 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0034-0534 at RA=0.5727h, DEC=-5.5769°
2025-10-07 20:14:39.675 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0218+4232 at RA=2.3018h, DEC=42.5382°
2025-10-07 20:14:39.675 | INFO     | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:569 - Discovered 3 unique pulsars across all PTAs
2025-10-07 20:14:39.676 | INFO     | metapulsar.metapulsar_factory:create_all_metapulsars:361 - Creating MetaPulsars for 3 pulsars
2025-10-07 20:14:39.676 | INFO     | metapulsar.metapulsa

   Option 1: Auto-select reference PTA by timespan (per pulsar)


2025-10-07 20:14:43.943 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/model_builder.py:225: UserWarning: Unrecognized parfile line 'EPHVER 5'
  warnings.warn(f"Unrecognized parfile line '{p_line}'", UserWarning)
2025-10-07 20:14:43.971 | WARNING  | pint.models.model_builder:__call__:229 - UNITS is not specified. Assuming TDB...
2025-10-07 20:14:47.433 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:14:47.434 | DEBUG    | metapulsar.pint_helpers:get_parameters_by_type_from_parfiles:216 - Component astrometry: Found 6 canonical parameters, 8 total with aliases
2025-10-07 20:14:47.461 | WARNING  | pint.models.model_builder:__call__:229 - UNITS is not specified. Assuming TDB...
2025-10-07 20:14:50.967 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovere

Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.2639.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/NRT.BON.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/NRT.BON.1600.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451

2025-10-07 20:15:02.331 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/timing_model.py:400: UserWarning: PINT does not support 'DILATEFREQ Y'
  warn("PINT does not support 'DILATEFREQ Y'")
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/timing_model.py:403: UserWarning: PINT only supports 'TIMEEPH FB90'
  warn("PINT only supports 'TIMEEPH FB90'")
2025-10-07 20:15:02.359 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `TimingModel` object should not be used for anything except converting to TDB.
/opt/venvs/pta/lib/python3.10/site-packages/pint/models/model_builder.py:225: UserWarning: Unrecognized parfile line 'DMASSPLANET1 0'
  warnings.warn(f"Unrecognized parfile line '{p_line}'", UserWarning)
/opt/venvs/pta/lib/python3.10

...

2025-10-07 20:15:39.415 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:15:39.472 | WARNING  | pint.models.model_builder:choose_binary_model:627 - Found T2 binary model. Gracefully converting T2 to: ELL1.
2025-10-07 20:15:39.475 | DEBUG    | pint.models.model_builder:convert_binary_params_dict:1056 - Requested to convert binary model for BINARY model: {binary}
2025-10-07 20:15:39.483 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `TimingModel` object should not be used for anything except converting to TDB.
2025-10-07 20:15:39.488 | WARNING  | pint.models.tcb_conversion:convert_tcb_tdb:133 - Converting this timing model from TCB to TDB. Please note that the TCB to TDB conversion is only approximate and the resulting timing model should be re-fit to get reliable results.
20

Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/JBO.DFB.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/JBO.DFB.1520.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/NRT.BON.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/



Results for PSR J0218+4232


RMS pre-fit residual = 0.000 (us), RMS post-fit residual = 125.667 (us)
Fit Chisq = 0	Chisqr/nfree = 0.00/0 = nan	pre/post = 0
Number of fit parameters: 0
Number of points in fit = 0
Offset: 0 1 offset_e*sqrt(n) = 0 n = 0


PARAMETER       Pre-fit                   Post-fit                  Uncertainty   Difference   Fit
---------------------------------------------------------------------------------------------------
RAJ (rad)       0.602600906916711         0.602600906916711         1.9037e-09    0             Y
RAJ (hms)       02:18:06.3572873           02:18:06.3572873         2.6177e-05    0            
DECJ (rad)      0.742430981883298         0.742430981883298         2.3023e-09    0             Y
DECJ (dms)      +42:32:17.38263           +42:32:17.38263           0.00047487    0            
F0 (s^-1)       430.461054545763          430.461054545763          4.3818e-12    0             Y
F1 (s^-2)       -1.43410127864114e-14     -1.43410127864114e

2025-10-07 20:16:21.933 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:16:21.972 | WARNING  | pint.models.model_builder:choose_binary_model:627 - Found T2 binary model. Gracefully converting T2 to: ELL1.
2025-10-07 20:16:21.973 | DEBUG    | pint.models.model_builder:convert_binary_params_dict:1056 - Requested to convert binary model for BINARY model: {binary}
2025-10-07 20:16:21.981 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `TimingModel` object should not be used for anything except converting to TDB.
2025-10-07 20:16:21.982 | WARNING  | pint.models.tcb_conversion:convert_tcb_tdb:133 - Converting this timing model from TCB to TDB. Please note that the TCB to TDB conversion is only approximate and the resulting timing model should be re-fit to get reliable results.
20

   Created 3 MetaPulsars with auto-selected reference PTAs


In [19]:
# Option 2: Use specific reference PTA for all pulsars
print("\n   Option 2: Use specific reference PTA for all pulsars")
metapulsars_epta = factory.create_all_metapulsars(
    file_data=limited_file_data,
    combination_strategy="consistent",
    reference_pta="EPTA_v2.2",  # EPTA as reference for all pulsars
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
)

print(f"   Created {len(metapulsars_epta)} MetaPulsars with EPTA as reference PTA")

2025-10-07 20:16:38.765 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 3 files for PTA discovered_EPTA_v2.2
2025-10-07 20:16:38.767 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0030+0451 at RA=0.5076h, DEC=4.8610°
2025-10-07 20:16:38.768 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0034-0534 at RA=0.5727h, DEC=-5.5769°
2025-10-07 20:16:38.770 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found pulsar J0218+4232 at RA=2.3018h, DEC=42.5382°
2025-10-07 20:16:38.770 | INFO     | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:569 - Discovered 3 unique pulsars across all PTAs
2025-10-07 20:16:38.772 | INFO     | metapulsar.metapulsar_factory:create_all_metapulsars:361 - Creating MetaPulsars for 3 pulsars
2025-10-07 20:16:38.772 | INFO     | metapulsar.metapulsa


   Option 2: Use specific reference PTA for all pulsars


2025-10-07 20:16:43.244 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:16:43.270 | WARNING  | pint.models.model_builder:__call__:229 - UNITS is not specified. Assuming TDB...
2025-10-07 20:16:47.377 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:16:47.377 | DEBUG    | metapulsar.pint_helpers:get_parameters_by_type_from_parfiles:216 - Component astrometry: Found 6 canonical parameters, 8 total with aliases
2025-10-07 20:16:47.405 | WARNING  | pint.models.model_builder:__call__:229 - UNITS is not specified. Assuming TDB...
2025-10-07 20:16:51.331 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:16:51.332 | DEBUG    | metapulsar.pint_helpers:get_parameters_by_type_from_parfiles:216 - Component spindown: Found 3 canonical parameters, 3 total wi

Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/EFF.EBPP.2639.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/NRT.BON.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451/tims/NRT.BON.1600.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0030+0451/J0030+0451_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0030+0451

2025-10-07 20:17:03.446 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:17:03.472 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `TimingModel` object should not be used for anything except converting to TDB.
2025-10-07 20:17:03.473 | WARNING  | pint.models.tcb_conversion:convert_tcb_tdb:133 - Converting this timing model from TCB to TDB. Please note that the TCB to TDB conversion is only approximate and the resulting timing model should be re-fit to get reliable results.
2025-10-07 20:17:06.892 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:17:06.893 | DEBUG    | metapulsar.pint_helpers:get_parameters_by_type_from_parfiles:216 - Component astrometry: Found 6 canonical parameters, 8 total wi

...

2025-10-07 20:17:41.401 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:17:41.441 | WARNING  | pint.models.model_builder:choose_binary_model:627 - Found T2 binary model. Gracefully converting T2 to: ELL1.
2025-10-07 20:17:41.442 | DEBUG    | pint.models.model_builder:convert_binary_params_dict:1056 - Requested to convert binary model for BINARY model: {binary}
2025-10-07 20:17:41.449 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `TimingModel` object should not be used for anything except converting to TDB.
2025-10-07 20:17:41.450 | WARNING  | pint.models.tcb_conversion:convert_tcb_tdb:133 - Converting this timing model from TCB to TDB. Please note that the TCB to TDB conversion is only approximate and the resulting timing model should be re-fit to get reliable results.
20

Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/JBO.DFB.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/JBO.DFB.1520.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/tims/NRT.BON.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0218+4232/J0218+4232_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0218+4232/



Results for PSR J0218+4232


RMS pre-fit residual = 0.000 (us), RMS post-fit residual = 125.667 (us)
Fit Chisq = 0	Chisqr/nfree = 0.00/0 = nan	pre/post = 0
Number of fit parameters: 0
Number of points in fit = 0
Offset: 0 1 offset_e*sqrt(n) = 0 n = 0


PARAMETER       Pre-fit                   Post-fit                  Uncertainty   Difference   Fit
---------------------------------------------------------------------------------------------------
RAJ (rad)       0.602600906916711         0.602600906916711         1.9037e-09    0             Y
RAJ (hms)       02:18:06.3572873           02:18:06.3572873         2.6177e-05    0            
DECJ (rad)      0.742430981883298         0.742430981883298         2.3023e-09    0             Y
DECJ (dms)      +42:32:17.38263           +42:32:17.38263           0.00047487    0            
F0 (s^-1)       430.461054545763          430.461054545763          4.3818e-12    0             Y
F1 (s^-2)       -1.43410127864114e-14     -1.43410127864114e

2025-10-07 20:18:19.761 | DEBUG    | metapulsar.pint_helpers:get_parameter_aliases_from_pint:67 - Discovered 221 parameter aliases from PINT
2025-10-07 20:18:19.802 | WARNING  | pint.models.model_builder:choose_binary_model:627 - Found T2 binary model. Gracefully converting T2 to: ELL1.
2025-10-07 20:18:19.803 | DEBUG    | pint.models.model_builder:convert_binary_params_dict:1056 - Requested to convert binary model for BINARY model: {binary}
2025-10-07 20:18:19.813 | WARNING  | pint.models.timing_model:validate:426 - PINT does not support 'UNITS TCB' internally. Reading this par file nevertheless because the `allow_tcb` option was given. This `TimingModel` object should not be used for anything except converting to TDB.
2025-10-07 20:18:19.814 | WARNING  | pint.models.tcb_conversion:convert_tcb_tdb:133 - Converting this timing model from TCB to TDB. Please note that the TCB to TDB conversion is only approximate and the resulting timing model should be re-fit to get reliable results.
20

   Created 3 MetaPulsars with EPTA as reference PTA


In [ ]:
# Show results (limited to 3 pulsars for demonstration)
print("\nResults (showing first 3 pulsars):")
all_pulsars = list(metapulsars_auto.keys())
print(f"  Total pulsars found: {len(all_pulsars)}")
for pulsar_name in all_pulsars[:3]:  # Show first 3
    auto_ref = list(metapulsars_auto[pulsar_name].pulsars.keys())[0]
    epta_ref = list(metapulsars_epta[pulsar_name].pulsars.keys())[0]
    print(f"    {pulsar_name}: auto={auto_ref}, epta={epta_ref}")

print("\nReference PTA selection:")
print("  - reference_pta=None: Auto-select by longest timespan per pulsar")
print("  - reference_pta='epta_dr2': Use EPTA as reference for all pulsars")
print("  - Manual: Use reorder_ptas_for_pulsar() for specific pulsars")


## Summary

This notebook demonstrated the complete MetaPulsar workflow:

### Key Takeaways

1. **Manual Data Preparation**: For single pulsars, manually creating data structures provides full control and transparency

2. **Consistent Strategy**: Merges compatible parameters while preserving PTA-specific information through clear naming conventions

3. **Enterprise Integration**: MetaPulsars are fully compatible with Enterprise analysis tools

4. **Automated Processing**: Pattern-based discovery enables processing of large pulsar arrays

5. **Flexible Reference PTA Selection**: Multiple strategies for choosing reference PTAs based on your analysis needs

### Next Steps

- Explore different combination strategies (`independent` vs `consistent`)
- Experiment with different component merging options
- Use the resulting MetaPulsars in Enterprise likelihood analysis
- Process larger pulsar arrays with automated discovery

The MetaPulsar framework provides a powerful and flexible way to combine multi-PTA pulsar timing data for gravitational wave detection and other analyses.
